In [110]:
#importing the necessery libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

In [111]:
train_data = pd.read_csv(r"C:\Users\Maedeh\Downloads\train.csv")
test_data = pd.read_csv(r"C:\Users\Maedeh\Downloads\test.csv")

see what we have ! 

In [112]:
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [113]:
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [114]:
train_data.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [115]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


data cleaning for both test and train data

In [116]:
train_encoded = pd.get_dummies(train_data,columns = ['Sex','Embarked'])
test_encoded = pd.get_dummies(test_data,columns = ['Sex','Embarked'])

In [117]:
train_data['Ticket'].nunique() / len(train_data)

0.7643097643097643

In [118]:
train_data['Cabin'].nunique()/len(train_data)

0.16498316498316498

In [119]:
train_data['Cabin'].isna().sum()

np.int64(687)

In [120]:
#adding 2 col from the data of cabin
train_encoded['Has_cabin']= train_data['Cabin'].notnull().astype(int)
test_encoded['Has_cabin']= test_data['Cabin'].notnull().astype(int)
train_encoded['deck'] = train_data['Cabin'].str[0]
test_encoded['deck'] = test_data['Cabin'].str[0]
train_encoded.drop(columns=['Cabin'] , inplace= True)
test_encoded.drop(columns=['Cabin'] , inplace= True)
train_encoded = pd.get_dummies(train_encoded , columns=['deck'])
test_encoded = pd.get_dummies(test_encoded , columns=['deck'])

In [121]:
train_encoded['Group_tic'] = train_data.groupby('Ticket')['Ticket'].transform('count')
test_encoded['Group_tic'] = test_data.groupby('Ticket')['Ticket'].transform('count')
train_encoded.drop(columns=['Ticket'] , inplace= True)
test_encoded.drop(columns=['Ticket'] , inplace= True)

In [122]:
train_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Age          714 non-null    float64
 5   SibSp        891 non-null    int64  
 6   Parch        891 non-null    int64  
 7   Fare         891 non-null    float64
 8   Sex_female   891 non-null    bool   
 9   Sex_male     891 non-null    bool   
 10  Embarked_C   891 non-null    bool   
 11  Embarked_Q   891 non-null    bool   
 12  Embarked_S   891 non-null    bool   
 13  Has_cabin    891 non-null    int64  
 14  deck_A       891 non-null    bool   
 15  deck_B       891 non-null    bool   
 16  deck_C       891 non-null    bool   
 17  deck_D       891 non-null    bool   
 18  deck_E       891 non-null    bool   
 19  deck_F  

In [123]:
train_encoded.head()

,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Fare,Sex_female,Sex_male,...,Has_cabin,deck_A,deck_B,deck_C,deck_D,deck_E,deck_F,deck_G,deck_T,Group_tic
0,1,0,3,"Braund, Mr. Owen Harris",22.0,1,0,7.2500,False,True,...,0,False,False,False,False,False,False,False,False,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,0,71.2833,True,False,...,1,False,False,True,False,False,False,False,False,1
2,3,1,3,"Heikkinen, Miss. Laina",26.0,0,0,7.9250,True,False,...,0,False,False,False,False,False,False,False,False,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,53.1000,True,False,...,1,False,False,True,False,False,False,False,False,2
4,5,0,3,"Allen, Mr. William Henry",35.0,0,0,8.0500,False,True,...,0,False,False,False,False,False,False,False,False,1


In [124]:
# handelling missing values in age
train_encoded['Age'] = train_encoded['Age'].fillna(train_encoded['Age'].median())
test_encoded['Age'] = test_encoded['Age'].fillna(test_encoded['Age'].median())

In [125]:
train_encoded.drop(columns=['Sex_male'],inplace=True)
test_encoded.drop(columns=['Sex_male'],inplace=True)
train_encoded.drop(columns=['Embarked_Q'], inplace=True)
test_encoded.drop(columns=['Embarked_Q'], inplace=True)
train_encoded.drop(columns=['deck_T'], inplace=True)
bc = train_encoded.select_dtypes(include=bool).columns
train_encoded[bc] = train_encoded[bc].astype(int)
bc_test = test_encoded.select_dtypes(include=bool).columns
test_encoded[bc_test] = test_encoded[bc_test].astype(int)

In [126]:
train_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 20 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Age          891 non-null    float64
 5   SibSp        891 non-null    int64  
 6   Parch        891 non-null    int64  
 7   Fare         891 non-null    float64
 8   Sex_female   891 non-null    int64  
 9   Embarked_C   891 non-null    int64  
 10  Embarked_S   891 non-null    int64  
 11  Has_cabin    891 non-null    int64  
 12  deck_A       891 non-null    int64  
 13  deck_B       891 non-null    int64  
 14  deck_C       891 non-null    int64  
 15  deck_D       891 non-null    int64  
 16  deck_E       891 non-null    int64  
 17  deck_F       891 non-null    int64  
 18  deck_G       891 non-null    int64  
 19  Group_ti

defining x and y for training

In [127]:
y = train_encoded['Survived']
X = train_encoded.drop(columns=['Name','PassengerId','Survived'])
x_train , x_test, y_train,y_test = train_test_split(X,y,random_state=42,test_size=0.2)

In [128]:
linear = LinearRegression()
linear.fit(x_train,y_train)
linear_pred = linear.predict(x_test)
print(linear.score(x_test,y_test))

0.43213944943366955


In [129]:
rf = RandomForestClassifier(n_estimators=50,random_state=42)
rf.fit(x_train , y_train)
print(rf.score(x_test,y_test))

0.7988826815642458


In [130]:
lr = LogisticRegression()
lr.fit(x_train , y_train)
lr_pred=lr.predict(x_test)
print(accuracy_score(y_test , lr_pred))

0.8100558659217877


C:\Users\Maedeh\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [131]:
model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
model.fit(x_train , y_train)
y_xgb = model.predict(x_test)
print(accuracy_score(y_test , y_xgb))

C:\Users\Maedeh\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [23:33:42] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


0.8156424581005587


In [132]:
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(x_train,y_train)
y_knn = knn.predict(x_test)
print(accuracy_score(y_test , y_knn))

0.7262569832402235


by testing different models it seems like xgbosst is the best choice

In [133]:
test_encoded.drop(columns=['Name','PassengerId'],inplace=True)
final_pred = model.predict(test_encoded)

In [134]:
submission = pd.DataFrame({
    "PassengerId": test_data["PassengerId"],
    "Survived": final_pred
})

submission.to_csv("submission.csv", index=False)

In [135]:
submission.to_csv("C:/Users/Maedeh/Desktop/submission.csv", index=False)